In [4]:
# """
# Figure 1

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.path as mpath
from matplotlib.patches import Patch, Circle
from matplotlib.lines import Line2D

import cartopy.crs as ccrs
import cartopy.feature as cfeature

# --------------------------------------------------------------------------
# Colour-blind-friendly palette (Okabe-Ito)
# --------------------------------------------------------------------------
COL = {
    "GBI": "#0072B2",  # blue
    "UBI": "#E69F00",  # orange
    "BHI": "#009E73",  # bluish green
    "SHI": "#CC79A7",  # reddish purple
    "AAI": "#D55E00",  # vermillion (cap boundary)
    "PVI": "#000000",  # black (bold dashed circle)
}

DATA_CRS = ccrs.PlateCarree()


def box_edges(lon_min, lon_max, lat_min, lat_max, n=200):
    """Return lon/lat arrays tracing a lon-lat box with n points per edge,
    so that on a curved projection the edges follow parallels/meridians
    instead of appearing as straight chords."""
    lons = np.concatenate([
        np.linspace(lon_min, lon_max, n),          # bottom  (lat_min)
        np.full(n, lon_max),                        # right   (lon_max)
        np.linspace(lon_max, lon_min, n),          # top     (lat_max)
        np.full(n, lon_min),                        # left    (lon_min)
    ])
    lats = np.concatenate([
        np.full(n, lat_min),
        np.linspace(lat_min, lat_max, n),
        np.full(n, lat_max),
        np.linspace(lat_max, lat_min, n),
    ])
    return lons, lats


# Domain definitions: (lon_min, lon_max, lat_min, lat_max, label_lon, label_lat)
BOXES = {
    "GBI": (-80, -20, 60.0, 80.0, -50, 70.0),
    "UBI": (-10,  80, 45.0, 80.0,  35, 62.0),
    "BHI": (-180, -120, 72.5, 82.5, -150, 77.5),
    "SHI": (80,  120, 40.0, 65.0, 100, 52.5),
}

FULLNAME = {
    "GBI": "Greenland Blocking Index (60–80°N, 20–80°W)",
    "UBI": "Ural Blocking Index (45–80°N, 10°W–80°E)",
    "BHI": "Beaufort High Index (72.5–82.5°N, 180–120°W)",
    "SHI": "Siberian High Index (40–65°N, 80–120°E)",
    "AAI": "Arctic Amplification Index (poleward of 70°N)",
    "PVI": "Polar Vortex Index (zonal-mean U, 10 hPa, 60°N)",
}

# --------------------------------------------------------------------------
# Figure & axes
# --------------------------------------------------------------------------
proj = ccrs.NorthPolarStereo(central_longitude=0)
plt.rcParams["hatch.linewidth"] = 0.6

fig = plt.figure(figsize=(9.7, 10.6))
# Axes inset from the figure edges to leave a margin ring for the (larger)
# longitude labels drawn just outside the circular boundary.
ax = fig.add_axes([0.085, 0.195, 0.83, 0.76], projection=proj)
ax.set_extent([-180, 180, 35, 90], DATA_CRS)
OUTER_LAT = 35.0  # map outer extent (circular boundary)

# Circular boundary
theta = np.linspace(0, 2 * np.pi, 200)
circle = mpath.Path(np.column_stack([np.sin(theta), np.cos(theta)]) * 0.5 + 0.5)
ax.set_boundary(circle, transform=ax.transAxes)

# Base map: land (very light grey) + coastlines (thin grey)
try:
    ax.add_feature(cfeature.LAND, facecolor="#f2f2f2", edgecolor="none", zorder=0)
    ax.coastlines(resolution="110m", linewidth=0.4, color="0.55", zorder=2.5)
    BASEMAP = True
except Exception as exc:  # offline: Natural Earth data unavailable
    print("WARNING: could not load Natural Earth basemap:", exc)
    BASEMAP = False

# Dotted graticules; latitude circles at 40/60/80 N
gl = ax.gridlines(
    draw_labels=False,
    xlocs=np.arange(-180, 181, 30),
    ylocs=[40, 60, 80],
    color="0.5", linestyle=":", linewidth=0.6, zorder=2,
)

# Manual latitude labels (robust on polar projections). 40/80 N on the 135 E
# radial; the 60 N label is moved along its circle into the box-free 160 E
# sector (away from the SHI corner) with an opaque halo, so it reads as a clean
# break in the coincident bold dashed PVI ring rather than sitting on it.
LAT_LABEL_LON = {40: 135, 60: 160, 80: 135}
for lat in (40, 60, 80):
    ax.text(LAT_LABEL_LON[lat], lat, f"{lat}°N", transform=DATA_CRS,
            ha="center", va="center", fontsize=7.5, color="0.3", zorder=6,
            bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none",
                      alpha=1.0 if lat == 60 else 0.8))

# Longitude labels as a ring just OUTSIDE the circular boundary, each placed
# radially at its own meridian (a few degrees south of the outer extent).
LON_LABELS = [
    (0, "0°"), (30, "30°E"), (60, "60°E"), (90, "90°E"), (120, "120°E"),
    (150, "150°E"), (180, "180°"), (-150, "150°W"), (-120, "120°W"),
    (-90, "90°W"), (-60, "60°W"), (-30, "30°W"),
]
lon_label_lat = OUTER_LAT - 3.5  # just outside the 35 N boundary
for lon, txt in LON_LABELS:
    ax.text(lon, lon_label_lat, txt, transform=DATA_CRS,
            ha="center", va="center", fontsize=10.5, color="0.4",
            zorder=6, clip_on=False)

# --------------------------------------------------------------------------
# AAI: polar cap poleward of 70 N  (UNDER the boxes)
# --------------------------------------------------------------------------
# A parallel projects to a circle centred on the pole; filling a pole-
# encircling lon/lat polygon breaks cartopy, so draw the cap as a native
# Circle in projected (data) coordinates instead.
x70, y70 = proj.transform_point(0, 70.0, DATA_CRS)
r70 = float(np.hypot(x70, y70))
# Light fill but denser/darker dot hatch so the cap reads at a glance while
# staying behind the boxes (per-colour alpha; no patch-level alpha).
ax.add_patch(Circle((0, 0), r70, transform=ax.transData,
                    facecolor=(0.6, 0.6, 0.6, 0.12),
                    edgecolor=(0.42, 0.42, 0.42, 0.55),
                    linewidth=0.0, hatch="......", zorder=1))
# Distinct boundary line at 70 N (a plotted ring transforms fine).
cap_lon = np.linspace(-180, 180, 400)
ax.plot(cap_lon, np.full_like(cap_lon, 70.0), transform=DATA_CRS,
        color=COL["AAI"], linewidth=1.8, linestyle=(0, (6, 3)), zorder=1.5)
ax.text(-135, 70.0, "AAI", transform=DATA_CRS, ha="center", va="center",
        fontsize=9, fontweight="bold", color=COL["AAI"], zorder=6,
        bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=COL["AAI"], alpha=0.85, lw=0.6))
# Ring label for the 70 N cap boundary (left side, clear of the 135 E graticule labels).
ax.text(-100, 70.0, "70°N", transform=DATA_CRS, ha="center", va="center",
        fontsize=7.5, fontweight="bold", color=COL["AAI"], zorder=6,
        bbox=dict(boxstyle="round,pad=0.15", fc="white", ec="none", alpha=0.85))

# --------------------------------------------------------------------------
# Index boxes: filled semi-transparent + solid coloured edge
# --------------------------------------------------------------------------
for key, (lo0, lo1, la0, la1, tlon, tlat) in BOXES.items():
    lons, lats = box_edges(lo0, lo1, la0, la1, n=200)
    ax.fill(lons, lats, transform=DATA_CRS,
            facecolor=COL[key], alpha=0.22, edgecolor="none", zorder=3)
    ax.plot(lons, lats, transform=DATA_CRS,
            color=COL[key], linewidth=2.0, solid_joinstyle="round", zorder=4)
    ax.text(tlon, tlat, key, transform=DATA_CRS, ha="center", va="center",
            fontsize=10, fontweight="bold", color=COL[key], zorder=6,
            bbox=dict(boxstyle="round,pad=0.2", fc="white", ec=COL[key],
                      alpha=0.9, lw=0.6))

# --------------------------------------------------------------------------
# PVI: bold dashed circle along 60 N, with annotation
# --------------------------------------------------------------------------
pvi_lon = np.linspace(-180, 180, 400)
pvi_lat = np.full_like(pvi_lon, 60.0)
# Drawn UNDER the index boxes (zorder < box fill=3 / edge=4), like the AAI cap,
# so GBI/UBI edges sit on top of it. Stays at exactly 60 N.
ax.plot(pvi_lon, pvi_lat, transform=DATA_CRS,
        color=COL["PVI"], linewidth=2.4, linestyle=(0, (7, 4)), zorder=2.6)

# Annotation sits in the upper-left sector (~150 W), in the box-free band
# between the outer boundary and the ring (BHI lies poleward of the ring here,
# so no polygon is crossed). Short radial arrow to the dashed ring.
x_on, y_on = proj.transform_point(-150, 60, DATA_CRS)    # point on the 60 N ring
x_tx, y_tx = proj.transform_point(-150, 48, DATA_CRS)    # empty band, boundary..ring
ax.annotate(
    "PVI: zonal-mean zonal wind\nat 10 hPa, 60°N",
    xy=(x_on, y_on), xytext=(x_tx, y_tx), textcoords="data",
    fontsize=10.5, ha="center", va="center", color="black", zorder=7,
    arrowprops=dict(arrowstyle="->", color="black", lw=1.1),
    bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="black", lw=0.7, alpha=0.9),
    clip_on=False,
)

# --------------------------------------------------------------------------
# Legend: full name + coordinate range
# --------------------------------------------------------------------------
handles = [
    Patch(facecolor=COL[k], edgecolor=COL[k], alpha=0.55,
          label=f"{k} — {FULLNAME[k]}")
    for k in ("GBI", "UBI", "BHI", "SHI")
]
handles.append(
    Patch(facecolor="#dddddd", edgecolor=COL["AAI"], hatch="....",
          label=f"AAI — {FULLNAME['AAI']}")
)
handles.append(
    Line2D([0], [0], color=COL["PVI"], lw=2.4, linestyle=(0, (7, 4)),
           label=f"PVI — {FULLNAME['PVI']}")
)

leg = fig.legend(
    handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0.012),
    ncol=2, frameon=True, fontsize=10.5, handlelength=2.0, columnspacing=1.6,
    borderpad=0.8, labelspacing=0.6, title="Arctic atmospheric index domains",
    title_fontsize=11.5,
)
leg.get_frame().set_edgecolor("0.6")
leg.get_frame().set_linewidth(0.7)

# --------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------
out = "fig1_index_domains"
fig.savefig(f"{out}.png", dpi=300, bbox_inches="tight", pad_inches=0.28, facecolor="white")
fig.savefig(f"{out}.pdf", bbox_inches="tight", pad_inches=0.28, facecolor="white")
print(f"Saved {out}.png and {out}.pdf  (basemap={'yes' if BASEMAP else 'NO'})")


Saved fig1_index_domains.png and fig1_index_domains.pdf  (basemap=yes)
